In [21]:
import pandas
import numpy
import seaborn as sns
import matplotlib as plt
from sklearn.preprocessing import RobustScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split
import joblib
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, accuracy_score, classification_report
from sklearn.metrics import recall_score, f1_score

In [22]:
X = pandas.read_csv("train.csv")
y = X["Credit_Score"]

C:\Users\sherr\AppData\Local\Temp\ipykernel_4484\1644057731.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  X = pandas.read_csv("train.csv")


In [ ]:
mode_value = X["Payment_Behaviour"].mode()[0]
X.loc[X["Payment_Behaviour"] == "3" , "Payment_Behaviour"]
X.head(2)

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.82262,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.94496,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good


In [24]:
X.drop(columns=["Credit_History_Age", "Name" , "Month","Customer_ID","Num_of_Delayed_Payment", "Credit_Mix"],inplace=True)

In [25]:

X["Type_of_Loan"] = X["Type_of_Loan"].fillna(X["Type_of_Loan"].mode())
X["Type_of_Loan"].isna().sum()
X["Num_Credit_Inquiries"] = X["Num_Credit_Inquiries"].fillna(X["Num_Credit_Inquiries"].median())
X["Monthly_Inhand_Salary"] = X["Monthly_Inhand_Salary"].fillna(X["Monthly_Inhand_Salary"].mean())
X.loc[X["Type_of_Loan"].isin(["Not Specified"]), "Type_of_Loan"] = X["Type_of_Loan"].mode()
X["Type_of_Loan"] = X["Type_of_Loan"].fillna(X["Type_of_Loan"].mode()[0])
X["Monthly_Balance"] = pandas.to_numeric(X["Monthly_Balance"] , errors="coerce")
X["Monthly_Balance"] = X["Monthly_Balance"].fillna(X["Monthly_Balance"].mean())
X["Monthly_Inhand_Salary"].isna().sum()
X["Outstanding_Debt"] = X["Outstanding_Debt"].astype(str).str.replace("_" , "")
X["Outstanding_Debt"] = pandas.to_numeric(X["Outstanding_Debt"], errors="coerce")
X["Changed_Credit_Limit"] = pandas.to_numeric(
    X["Changed_Credit_Limit"].replace("_", numpy.nan),
    errors="coerce"
)

X["Changed_Credit_Limit"] = X["Changed_Credit_Limit"].fillna(
    X["Changed_Credit_Limit"].mean()
)
X["Annual_Income"] = X["Annual_Income"].astype(str).str.replace("_" ,"")
X["Annual_Income"] = pandas.to_numeric(X["Annual_Income"] , errors="coerce")
X["Num_of_Loan"] = X["Num_of_Loan"].astype(str).str.replace("_" , "")
X["Num_of_Loan"] = pandas.to_numeric(X["Num_of_Loan"] , errors="coerce")
X.isna().sum()

ID                             0
Age                            0
SSN                            0
Occupation                     0
Annual_Income                  0
Monthly_Inhand_Salary          0
Num_Bank_Accounts              0
Num_Credit_Card                0
Interest_Rate                  0
Num_of_Loan                    0
Type_of_Loan                   0
Delay_from_due_date            0
Changed_Credit_Limit           0
Num_Credit_Inquiries           0
Outstanding_Debt               0
Credit_Utilization_Ratio       0
Payment_of_Min_Amount          0
Total_EMI_per_month            0
Amount_invested_monthly     4479
Payment_Behaviour              0
Monthly_Balance                0
Credit_Score                   0
dtype: int64

In [26]:
X["Outstanding_Debt"] = X["Outstanding_Debt"].astype(str).str.replace("_" , "")
X["Outstanding_Debt"] = pandas.to_numeric(X["Outstanding_Debt"] , errors="coerce")
X["Annual_Income"].dtype

dtype('float64')

In [27]:
X["Amount_invested_monthly"] = X["Amount_invested_monthly"].astype(str).str.replace("__" , "")
X["Amount_invested_monthly"] = pandas.to_numeric(X["Amount_invested_monthly"] , errors="coerce")

In [28]:
robust_cols = [
    "Outstanding_Debt",
    "Annual_Income"
]
standard_cols = [
    "Changed_Credit_Limit",
    "Interest_Rate",
    "Credit_Utilization_Ratio",
    "Total_EMI_per_month",
    "Amount_invested_monthly",
    "Monthly_Balance",
    "Delay_from_due_date",
    "Num_Credit_Inquiries",
    "Num_Credit_Card",
    "Num_of_Loan",
    "Num_Bank_Accounts"
]

categorical_cols = [
    "Occupation",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
    "Type_of_Loan"
]
robust_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])
standard_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

processing = ColumnTransformer([
    ("robust", robust_pipeline, robust_cols),
    ("standard", standard_pipeline, standard_cols),
    ("cat", categorical_pipeline, categorical_cols)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
pipeline = Pipeline([
    ("processing", processing),
    ("model", LogisticRegression(max_iter=2000))
])

# Train model
print("Training model on 100k rows...")
pipeline.fit(X_train, y_train)
print("Training complete!")

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluation
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Weighted Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training model on 100k rows...


c:\Users\sherr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training complete!

RESULTS
Accuracy: 0.7080
Weighted Precision: 0.7072

Classification Report:
              precision    recall  f1-score   support

        Good       0.59      0.45      0.51      3527
        Poor       0.79      0.67      0.73      5874
    Standard       0.70      0.81      0.75     10599

    accuracy                           0.71     20000
   macro avg       0.69      0.65      0.66     20000
weighted avg       0.71      0.71      0.70     20000



In [29]:
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")


Recall: 0.71
F1 Score: 0.70


In [30]:
joblib.dump(pipeline, "credit_risk_pipeline.pkl")

print("Pipeline saved successfully!")


Pipeline saved successfully!
